# 03 — Q4: Tax burden comparison

**Question:** What's the property tax delta between Lawrence Township / Hamilton / Pennington / Princeton Jct for a comparable home?

**Data:** `taxes` and `demographics` tables (Census ACS 5-year 2022).

**Caveat:** Census top-codes taxes at $10,001 for privacy. Pennington and Princeton Jct will be marked as topcoded in the data — their real taxes are *higher* than shown, so effective rates for those two are **lower bounds**.

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

DB = Path.cwd().parent / 'data' / 'clean' / 'housing.db'
ZIP_NAMES = {
    '08534': 'Pennington',
    '08550': 'Princeton Jct',
    '08619': 'Hamilton',
    '08648': 'Lawrenceville',
}

# Join demographics and taxes — one row per ZIP with everything we need
conn = sqlite3.connect(DB)
q4 = pd.read_sql("""
    SELECT
        d.zip,
        d.median_home_value,
        d.median_hh_income,
        t.median_real_estate_taxes AS annual_taxes,
        t.effective_tax_rate_pct AS tax_rate_pct,
        t.taxes_topcoded,
        ROUND(t.median_real_estate_taxes * 100.0 / d.median_hh_income, 2) AS tax_pct_of_income
    FROM demographics d
    JOIN taxes t USING (zip)
    ORDER BY tax_rate_pct DESC
""", conn)
conn.close()
q4['name'] = q4['zip'].map(ZIP_NAMES)
q4

In [ ]:
# Bar chart — effective tax rates per ZIP, topcoded bars marked with hatching
fig, ax = plt.subplots()
colors = {'Pennington': '#2E86AB', 'Princeton Jct': '#A23B72', 'Hamilton': '#F18F01', 'Lawrenceville': '#C73E1D'}

bars = ax.bar(
    q4['name'],
    q4['tax_rate_pct'],
    color=[colors.get(n, 'gray') for n in q4['name']],
    edgecolor='black',
    linewidth=1,
)

# Hatch the top-coded bars so viewers know they're lower bounds
for bar, topcoded in zip(bars, q4['taxes_topcoded']):
    if topcoded:
        bar.set_hatch('///')

# Annotate bars with actual values
for bar, rate, taxes, topcoded in zip(bars, q4['tax_rate_pct'], q4['annual_taxes'], q4['taxes_topcoded']):
    label = f'{rate:.2f}%\n${taxes:,}/yr'
    if topcoded:
        label += '\n(lower bound)'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, label,
            ha='center', va='bottom', fontsize=9)

ax.set_title('Effective Property Tax Rate by ZIP (Census ACS 2022)', fontsize=13, pad=10)
ax.set_ylabel('Annual taxes as % of median home value')
ax.set_ylim(0, max(q4['tax_rate_pct']) * 1.4)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scenario: what would YOU pay for a $500K home in each ZIP?
# Uses the effective rate (tax / home value), applied to a $500K target.
# For topcoded ZIPs, this is a LOWER BOUND — real taxes would be higher.
HOME_PRICE = 500_000

scenario = q4[['name', 'zip', 'tax_rate_pct', 'taxes_topcoded']].copy()
scenario[f'annual_tax_on_${HOME_PRICE:,}'] = (scenario['tax_rate_pct'] / 100 * HOME_PRICE).round().astype(int)
scenario[f'monthly_tax_on_${HOME_PRICE:,}'] = (scenario[f'annual_tax_on_${HOME_PRICE:,}'] / 12).round().astype(int)
scenario = scenario.sort_values(f'annual_tax_on_${HOME_PRICE:,}')

print(f"Projected tax on a ${HOME_PRICE:,} home, using each ZIP's effective rate:")
scenario

## Finding — Q4

The four ZIPs span a wide tax-burden range. On a hypothetical **$500K
home**, annual property tax by ZIP:

| ZIP | Name | Effective rate | Annual tax on $500K | Monthly |
|---|---|---|---|---|
| 08619 | Hamilton | 2.72% | **$13,600** | $1,133 |
| **08648** | **Lawrenceville** | **2.28%** | **$11,400** | **$950** |
| 08534 | Pennington | ≥1.83% | ≥$9,150 | ≥$762 |
| 08550 | Princeton Jct | ≥1.40% | ≥$7,000 | ≥$583 |

**Two honest caveats** before drawing conclusions:

1. **Pennington and Princeton Jct rates are lower bounds.** Census
   top-codes tax data at $10,001 for privacy — those two ZIPs hit the
   cap, so their real effective rates are higher than shown. The
   hatched bars in the chart are the visual reminder.
2. **Same effective rate applied to different home values gives very
   different absolute tax bills.** Princeton Jct's median home is
   $714K, not $500K — real tax there is almost certainly ≥$10K/year.

**The actionable finding for our move:**

- **Hamilton is the most expensive ZIP per dollar of home value** —
  2.72% of your home value paid in taxes every year. On a $500K house,
  that's **$2,200 MORE per year than Lawrenceville**, or **$66K more
  over a 30-year mortgage**.
- **Lawrenceville (08648) comes in at the median** — 2.28% effective,
  $11,400/yr on a $500K home.
- **Princeton Jct and Pennington look cheaper on effective rate**, but
  their homes are 40-60% more expensive, so absolute taxes are likely
  comparable or higher than Lawrenceville.

**Implication:** For a comparably-priced home in our budget range,
**Lawrenceville has a meaningful tax advantage over Hamilton** and
is roughly break-even with the premium ZIPs on total-tax basis. If
we stretch our budget to Princeton Jct, we trade tax rate savings for
a larger absolute tax bill driven by higher home value.